In [ ]:
print("="*70)
print("ArbItro - Pipeline 2 Test")
print("="*70)

import tensorflow as tf
print(f"\nTensorFlow version: {tf.__version__}")

gpu_name = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
if gpu_name:
    gpu_info = gpu_name[0].split(',')
    print(f"  GPU  : {gpu_info[0].strip()}")
    print(f"  VRAM : {gpu_info[1].strip()}")
    print("\n  System Resources:")
    !free -h | grep Mem
    !nvidia-smi --query-gpu=memory.free,memory.total --format=csv
else:
    print("\n  No GPU found")

print("\n" + "="*70)

In [ ]:
import os, sys

try:
    from google.colab import drive
    USE_COLAB = True
    drive.mount('/content/drive')
    BASE_DIR      = '/content/drive/MyDrive/ArbItro'
    CODE_DIR      = os.path.join(BASE_DIR, 'ArbItro_code')
    DATASET_DIR   = os.path.join(BASE_DIR, 'ArbItro_dataset', 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')
except ImportError:
    USE_COLAB = False
    BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
    CODE_DIR      = os.path.join(BASE_DIR, 'model', 'src', 'pipeline2')
    DATASET_DIR   = os.path.join(BASE_DIR, 'dataset')
    TRAINING_ROOT = os.path.join(BASE_DIR, 'ArbItro_Training')

sys.path.insert(0, CODE_DIR)

print(f"  USE_COLAB    : {USE_COLAB}")
print(f"  CODE_DIR     : {CODE_DIR}")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  TRAINING_ROOT: {TRAINING_ROOT}")

required_paths = [
    os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    os.path.join(DATASET_DIR, 'test'),
]
print("\nVerifying test paths:")
for path in required_paths:
    status = "Found" if os.path.exists(path) else "Not found"
    print(f"  {status}: {path}")

In [ ]:
!pip install opencv-python-headless

In [ ]:
if USE_COLAB:
    !mkdir -p /content/arbitro_model
    %cd /content/arbitro_model
    !cp "{CODE_DIR}/data_loader.py" .
    !cp "{CODE_DIR}/model.py" .
    sys.path.insert(0, '/content/arbitro_model')
    print("\nFiles copied to /content/arbitro_model:")
    !ls -lh

In [ ]:
try:
    from data_loader import ArbItroDataGenerator
    from model import (
        build_arbitro_model_speed_aware_lstm_multiclip,
        compile_arbitro_model,
        BinaryBalancedAccuracy,
        MulticlassBalancedAccuracy,
    )
    print("  Pipeline 2 imports successful!")
except Exception as e:
    print(f"  Import error: {e}")
    raise

In [ ]:
TEST_CONFIG = {
    'json_path'  : os.path.join(DATASET_DIR, 'test', 'annotations.json'),
    'video_dir'  : os.path.join(DATASET_DIR, 'test'),
    'model_path' : os.path.join(TRAINING_ROOT, 'models', 'pipeline2.keras'),
    'dim'        : (224, 398),
    'n_frames'   : 16,
    'max_clips'  : 4,
    'batch_size' : 28,
    'shuffle'    : False,
}

_OFF_MAP = {"No offence": 0, "": 0, "Between": 1, "Offence": 1}
_ACT_MAP = {
    "": 0, "Dont know": 0, "Dive": 0,
    "Challenge": 1, "Tackling": 1, "Standing tackling": 1,
    "Holding": 2, "Pushing": 2, "Elbowing": 2,
    "High leg": 3,
}
MAP_OFF = {0: "No Offence",   1: "Offence"}
MAP_SEV = {0: "No Card",      1: "Yellow",      2: "Red"}
MAP_ACT = {0: "Neutral/Dive", 1: "Tackles",     2: "Upper Body", 3: "High Leg"}

def _get_sev(row):
    try:
        v = float(row.get('Severity', 0))
        if v >= 5.0: return 2
        if v >= 3.0: return 1
        return 0
    except: return 0

status = "Found" if os.path.exists(TEST_CONFIG['model_path']) else "Not found"
print(f"  {status}: {TEST_CONFIG['model_path']}")